# Build Your First CNN in PyTorch: Fashion-MNIST Image Classification

**Beginner-friendly, Google Colab-ready tutorial — no local dataset needed.**

In this notebook, we will build a Convolutional Neural Network (CNN) that classifies small grayscale clothing images. `torchvision` will download Fashion-MNIST automatically when you run the dataset cell.

### What you will learn
- Load and inspect image data
- Understand image tensors and batches
- Build a small CNN
- Train it using loss, backpropagation, and an optimizer
- Evaluate accuracy and make predictions

**For recording:** Run one section at a time. Explain the markdown in simple words, type/run the code, and discuss the output before continuing.

**Colab setup:** Open this notebook in Google Colab. A GPU is optional but helpful: Runtime → Change runtime type → T4 GPU (if available). This notebook's code has not been executed here; you can run each cell yourself.


## 1. Import libraries

PyTorch builds and trains the network. `torchvision` provides the dataset and image transformations. Matplotlib displays images and graphs.

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


### What are these imports for?
- `torch`: tensors and core operations.
- `torch.nn`: neural-network layers and loss functions.
- `torch.optim`: optimizers that update model parameters.
- `DataLoader`: creates batches and can shuffle examples.
- `torchvision.datasets`: provides Fashion-MNIST.
- `transforms`: prepares images for the model.
- `matplotlib`: visualizes images and training progress.

Colab normally includes PyTorch and torchvision. If an import fails, read the error and check the runtime before installing or upgrading packages.

## 2. Choose CPU or GPU

This code selects a GPU if Colab provides one; otherwise it uses the CPU. The rest of the notebook works on either device.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3. Download and load Fashion-MNIST

Fashion-MNIST contains **60,000 training images** and **10,000 test images**. Each image is 28 × 28 pixels, grayscale, and belongs to one of 10 clothing categories.

The setting `download=True` downloads the data automatically into the Colab runtime. You do not need to provide a local dataset file. If the runtime is reset, the data may need to be downloaded again.

In [ ]:
# Convert images to tensors, then normalize pixel values to roughly [-1, 1].
# These images are grayscale, so there is one channel.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Number of classes:", len(class_names))


### Understand the transformation
`ToTensor()` converts an image into a PyTorch tensor and scales pixel values from 0–255 to 0–1. `Normalize((0.5,), (0.5,))` then maps them to approximately -1 to 1. Normalization can help training.

The label is a number from 0 to 9. `class_names` lets us display the corresponding clothing name.

## 4. View some training images

Before building a model, inspect the data. Because the images were normalized, we reverse that normalization for display.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax in axes.flat:
    index = random.randrange(len(train_dataset))
    image, label = train_dataset[index]
    image_to_show = image.squeeze(0) * 0.5 + 0.5  # Back to [0, 1] for display
    ax.imshow(image_to_show, cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Create batches with DataLoader

A **batch** is a group of images processed together. We use 64 images per batch rather than sending the whole dataset through the network at once. Shuffling training data changes the order of examples each epoch. For evaluation, shuffling is unnecessary.

In [ ]:
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch of images:", images.shape)
print("Batch of labels:", labels.shape)
print("First few labels:", labels[:10].tolist())


### Read the tensor shape
You should see an image batch shaped like `[64, 1, 28, 28]`:
- `64`: number of images in this batch
- `1`: one grayscale channel
- `28, 28`: image height and width

The labels tensor has one correct category for each image.

## 6. Build the CNN

Convolution layers learn filters that detect useful visual patterns. ReLU adds non-linearity. Max pooling reduces the height and width of feature maps. The final linear layer produces 10 scores, one for each clothing category.

| Stage | Output shape (channels × height × width) |
|---|---|
| Input | 1 × 28 × 28 |
| Conv 1: 16 filters, padding 1 | 16 × 28 × 28 |
| Max pool | 16 × 14 × 14 |
| Conv 2: 32 filters, padding 1 | 32 × 14 × 14 |
| Max pool | 32 × 7 × 7 |
| Flatten | 1,568 values |
| Output layer | 10 scores |

Padding of 1 preserves height and width for a 3 × 3 convolution. Each 2 × 2 max-pooling layer halves height and width.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Feature extractor: learns patterns such as edges, shapes, and textures.
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Classification head: converts learned feature maps into 10 class scores.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = FashionCNN().to(device)
print(model)


### What does `forward()` mean?
`forward()` describes how data travels through the network. First, the image goes through convolution and pooling layers. `Flatten()` turns the feature maps into a one-dimensional list for the final linear layer.

The model returns **raw class scores**, also called *logits*. `CrossEntropyLoss` uses these directly and handles the relevant calculation internally, so do not add a softmax layer before this loss.

## 7. Check one forward pass

Before training, pass one batch through the model. This checks that the input and output dimensions match expectations.

In [ ]:
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

model.eval()
with torch.no_grad():
    sample_outputs = model(sample_images)

print("Input batch shape:", sample_images.shape)
print("Output scores shape:", sample_outputs.shape)
print("Scores for the first image:", sample_outputs[0].cpu())
model.train()  # Switch back to training mode before training.


The output should have shape `[64, 10]`: 64 images and 10 scores per image. The scores are not meaningful yet because the model has not learned from the training data.

## 8. Choose the loss function and optimizer

- **Cross-entropy loss:** compares class scores with the correct labels. Lower is generally better.
- **Adam optimizer:** updates learnable parameters to reduce the loss.
- **Learning rate 0.001:** controls the size of parameter updates.

We have a 10-class classification problem, so cross-entropy is a sensible starting choice.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## 9. Train the CNN

We will train for five epochs to keep this first tutorial manageable. One **epoch** means the model has processed the complete training dataset once.

For each batch, the steps are:
1. Clear old gradients.
2. Make predictions with a forward pass.
3. Calculate the loss.
4. Use `loss.backward()` to calculate gradients through backpropagation.
5. Use `optimizer.step()` to update model parameters.

Training time depends on the runtime and whether a GPU is available.

In [ ]:
num_epochs = 5
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()              # 1. Clear gradients from the previous batch.
        outputs = model(images)            # 2. Forward pass: get class scores.
        loss = criterion(outputs, labels)  # 3. Compare scores with correct labels.
        loss.backward()                    # 4. Backpropagation: calculate gradients.
        optimizer.step()                   # 5. Update model parameters.

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_loss)
    print(f"Epoch [{epoch + 1}/{num_epochs}] - Training loss: {epoch_loss:.4f}")


### Interpret the loss
The printed value is the average training loss for that epoch. We generally hope it decreases over time. A falling training loss is useful, but it does **not** prove the model will do well on new images. We will measure performance on a separate test set.

## 10. Plot training loss

The graph helps us see how the training loss changed across epochs.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average training loss")
plt.title("Training Loss Across Epochs")
plt.xticks(range(1, num_epochs + 1))
plt.grid(True)
plt.show()


## 11. Evaluate on the test set

The test images were not used to update the model parameters. We measure how often the trained model predicts the correct category. `model.eval()` switches to evaluation mode, and `torch.no_grad()` avoids tracking gradients.

In [ ]:
model.eval()
correct = 0
total = 0
test_loss_total = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total
average_test_loss = test_loss_total / total
print(f"Test loss: {average_test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2f}% ({correct}/{total} correct)")


### What does test accuracy mean?
If accuracy were 88%, for example, the model would classify about 88 out of every 100 test images correctly. Your result may differ because of random initialization, training conditions, and runtime differences. Do not expect one exact percentage.

Some categories look similar—for example, a shirt, T-shirt, and pullover—so mistakes are normal.

## 12. Compare predictions with true labels

Here we display ten test images. Each title shows the model's predicted class, the actual class, and whether they match.

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    outputs = model(images.to(device))
    predictions = outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    image_to_show = images[i].squeeze(0) * 0.5 + 0.5
    predicted_name = class_names[predictions[i].item()]
    actual_name = class_names[labels[i].item()]
    is_correct = predictions[i].item() == labels[i].item()
    ax.imshow(image_to_show, cmap="gray")
    ax.set_title(
        f"Predicted: {predicted_name}\nActual: {actual_name}\n"
        f"{'Correct' if is_correct else 'Incorrect'}"
    )
    ax.axis("off")
plt.tight_layout()
plt.show()


## 13. Predict one randomly selected test image

The model returns ten scores. `argmax` selects the index of the largest score, which is our predicted class.

In [ ]:
index = random.randrange(len(test_dataset))
image, true_label = test_dataset[index]

model.eval()
with torch.no_grad():
    scores = model(image.unsqueeze(0).to(device))
    predicted_label = scores.argmax(dim=1).item()

plt.figure(figsize=(3, 3))
plt.imshow(image.squeeze(0) * 0.5 + 0.5, cmap="gray")
plt.axis("off")
plt.show()

print("Predicted category:", class_names[predicted_label])
print("Correct category:  ", class_names[true_label])


## 14. Recap and quick questions

In this notebook, we:
1. Loaded Fashion-MNIST directly through `torchvision`—no local dataset files were needed.
2. Converted images to tensors and created batches.
3. Built a CNN using convolution, ReLU, max pooling, flattening, and a linear output layer.
4. Trained it with cross-entropy loss, backpropagation, and Adam.
5. Evaluated test accuracy and inspected predictions.

### Ask viewers
1. Why is `in_channels=1` in the first convolution layer?
2. What does max pooling do to the feature maps?
3. Why do we call `optimizer.zero_grad()` before backpropagation?
4. Why evaluate on test data that was not used for training?
5. What does `argmax(dim=1)` do?

**Possible next video:** Improve CNN performance—inspect mistakes, understand overfitting, and introduce data augmentation and dropout.